# Day 3: Text Search

Now that we have documents (Day 1) and chunks (Day 2), we need a way to search them. We'll start with **text search** (also called lexical search), which uses exact keyword matching to find relevant chunks.

## What We're Building

A text search system that:
1. Indexes chunked documents using TF-IDF scoring
2. Supports field boosting (title matches rank higher than content matches)
3. Returns top-K results with relevance scores
4. Enables fast keyword-based retrieval

## Key Concepts

**TF-IDF (Term Frequency-Inverse Document Frequency)**: A statistical measure that evaluates how important a word is to a document in a collection. Words that appear frequently in a document but rarely across all documents get higher scores.

**Field Boosting**: Giving certain fields (like title) more weight than others (like content). A title match is often more relevant than a content match.

**Lexical vs Semantic Search**: Lexical search matches exact keywords ("machine learning" matches "machine learning"). Semantic search understands meaning ("ML" matches "machine learning"). We'll do semantic search on Day 4.

## Imports and Data Loading

We'll import functions from Day 1 (data ingestion) and Day 2 (chunking), then load and chunk our documents.

In [ ]:
# Import from previous days
from day1 import read_repo_data
from day2 import chunk_sliding_window

# Import minsearch for text search
import minsearch

print("✓ All imports successful")

In [ ]:
# Load DataTalksClub FAQ
print("Loading DataTalksClub FAQ...")
datatalk_docs = read_repo_data("DataTalksClub", "faq")
print(f"Loaded {len(datatalk_docs)} documents")

In [ ]:
# Load Evidently AI docs
print("Loading Evidently docs...")
evidently_docs = read_repo_data("evidentlyai", "docs")
print(f"Loaded {len(evidently_docs)} documents")

In [ ]:
# Chunk DataTalksClub FAQ using sliding window (2000 chars, 1000 overlap)
print("Chunking DataTalksClub FAQ...")
datatalk_chunks = []
for doc in datatalk_docs:
    chunks = chunk_sliding_window(doc, chunk_size=2000, overlap=1000)
    datatalk_chunks.extend(chunks)

print(f"Created {len(datatalk_chunks)} chunks from {len(datatalk_docs)} documents")

In [ ]:
# Chunk Evidently docs using sliding window
print("Chunking Evidently docs...")
evidently_chunks = []
for doc in evidently_docs:
    chunks = chunk_sliding_window(doc, chunk_size=2000, overlap=1000)
    evidently_chunks.extend(chunks)

print(f"Created {len(evidently_chunks)} chunks from {len(evidently_docs)} documents")

## Preparing Documents for Indexing

minsearch expects documents with consistent fields. We'll add a "title" field to each chunk (extracted from metadata or filename).

In [ ]:
def prepare_for_indexing(chunks):
    """Add title field to chunks for consistent indexing.
    
    Args:
        chunks: List of chunk dicts with metadata
    
    Returns:
        List of chunks with 'title' field added
    """
    prepared = []
    for chunk in chunks:
        # Extract title from metadata or use filename
        title = chunk['metadata'].get('title', chunk['filename'])
        
        # Create new chunk dict with title field
        prepared_chunk = {
            'content': chunk['content'],
            'title': title,
            'filename': chunk['filename'],
            'chunk_id': chunk['chunk_id'],
            'metadata': chunk['metadata']
        }
        prepared.append(prepared_chunk)
    
    return prepared

# Prepare both datasets
datatalk_prepared = prepare_for_indexing(datatalk_chunks)
evidently_prepared = prepare_for_indexing(evidently_chunks)

print(f"Prepared {len(datatalk_prepared)} DataTalksClub chunks")
print(f"Prepared {len(evidently_prepared)} Evidently chunks")
print(f"\nSample prepared chunk:")
print(f"  title: {datatalk_prepared[0]['title']}")
print(f"  filename: {datatalk_prepared[0]['filename']}")
print(f"  chunk_id: {datatalk_prepared[0]['chunk_id']}")
print(f"  content preview: {datatalk_prepared[0]['content'][:100]}...")

## Creating the Text Search Index

We'll use minsearch.Index with:
- **text_fields**: Fields to index for text search (content, title)
- **keyword_fields**: Fields for exact match filtering (filename, chunk_id)

Field boosting will be applied at search time (title:2.0, content:1.0).

In [ ]:
# Create index for DataTalksClub FAQ
print("Creating DataTalksClub FAQ index...")
datatalk_index = minsearch.Index(
    text_fields=["content", "title"],
    keyword_fields=["filename", "chunk_id"]
)

# Fit index with prepared chunks
datatalk_index.fit(datatalk_prepared)
print(f"✓ Indexed {len(datatalk_prepared)} DataTalksClub chunks")

In [ ]:
# Create index for Evidently docs
print("Creating Evidently docs index...")
evidently_index = minsearch.Index(
    text_fields=["content", "title"],
    keyword_fields=["filename", "chunk_id"]
)

# Fit index with prepared chunks
evidently_index.fit(evidently_prepared)
print(f"✓ Indexed {len(evidently_prepared)} Evidently chunks")

## text_search() Function

We'll create a wrapper function that provides a consistent interface for text search across both datasets.

Field boosting is applied via `boost_dict={"title": 2.0, "content": 1.0}` - title matches get 2x weight because a title match is often more relevant than a content match.

In [ ]:
def text_search(index, query: str, top_k: int = 5) -> list[dict]:
    """Search index using text/lexical matching with TF-IDF scoring and field boosting.
    
    Args:
        index: minsearch.Index instance
        query: Search query string
        top_k: Number of results to return (default 5)
    
    Returns:
        List of dicts with content, title, filename, chunk_id, metadata, score
        Sorted by relevance score (highest first)
    """
    results = index.search(
        query=query,
        boost_dict={"title": 2.0, "content": 1.0},
        num_results=top_k
    )
    return results

print("✓ text_search() function defined")

## Test on DataTalksClub FAQ

Let's run queries where text search excels: exact terminology, specific course names, and technology names.

In [ ]:
# Query 1: "machine learning" - exact terminology
print("=" * 80)
print("Query: 'machine learning'")
print("=" * 80)

results = text_search(datatalk_index, "machine learning", top_k=5)
print(f"\nFound {len(results)} results:\n")

for i, result in enumerate(results, 1):
    print(f"{i}. [Score: {result.get('score', 0):.3f}] {result['title']}")
    print(f"   File: {result['filename']}")
    print(f"   Preview: {result['content'][:150]}...")
    print()

In [ ]:
# Query 2: "data engineering" - exact course name
print("=" * 80)
print("Query: 'data engineering'")
print("=" * 80)

results = text_search(datatalk_index, "data engineering", top_k=5)
print(f"\nFound {len(results)} results:\n")

for i, result in enumerate(results, 1):
    print(f"{i}. [Score: {result.get('score', 0):.3f}] {result['title']}")
    print(f"   File: {result['filename']}")
    print(f"   Preview: {result['content'][:150]}...")
    print()

In [ ]:
# Query 3: "Docker" - specific technology name
print("=" * 80)
print("Query: 'Docker'")
print("=" * 80)

results = text_search(datatalk_index, "Docker", top_k=5)
print(f"\nFound {len(results)} results:\n")

for i, result in enumerate(results, 1):
    print(f"{i}. [Score: {result.get('score', 0):.3f}] {result['title']}")
    print(f"   File: {result['filename']}")
    print(f"   Preview: {result['content'][:150]}...")
    print()

### Why Text Search Works Here

Text search excels at:
- **Exact keyword matching**: "machine learning" matches documents containing exactly those words
- **Technical terminology**: "Docker", "data engineering" are precise terms with clear meanings
- **Proper names**: Course names, technology names, acronyms

The TF-IDF scoring ensures documents that use these terms frequently (but not too commonly across all documents) rank higher.

## Test on Evidently Docs

Let's test on technical documentation with domain-specific terminology.

In [ ]:
# Query 1: "monitoring" - exact technical term
print("=" * 80)
print("Query: 'monitoring'")
print("=" * 80)

results = text_search(evidently_index, "monitoring", top_k=5)
print(f"\nFound {len(results)} results:\n")

for i, result in enumerate(results, 1):
    print(f"{i}. [Score: {result.get('score', 0):.3f}] {result['title']}")
    print(f"   File: {result['filename']}")
    print(f"   Preview: {result['content'][:150]}...")
    print()

In [ ]:
# Query 2: "data drift" - specific concept
print("=" * 80)
print("Query: 'data drift'")
print("=" * 80)

results = text_search(evidently_index, "data drift", top_k=5)
print(f"\nFound {len(results)} results:\n")

for i, result in enumerate(results, 1):
    print(f"{i}. [Score: {result.get('score', 0):.3f}] {result['title']}")
    print(f"   File: {result['filename']}")
    print(f"   Preview: {result['content'][:150]}...")
    print()

In [ ]:
# Query 3: "evidently" - product name
print("=" * 80)
print("Query: 'evidently'")
print("=" * 80)

results = text_search(evidently_index, "evidently", top_k=5)
print(f"\nFound {len(results)} results:\n")

for i, result in enumerate(results, 1):
    print(f"{i}. [Score: {result.get('score', 0):.3f}] {result['title']}")
    print(f"   File: {result['filename']}")
    print(f"   Preview: {result['content'][:150]}...")
    print()

## Text Search Limitations

Let's see where text search struggles: paraphrases and conceptual queries.

In [ ]:
# Limitation 1: Paraphrase - "how to improve models" won't match "fine-tuning"
print("=" * 80)
print("Limitation Query: 'how to improve models'")
print("(Looking for content about fine-tuning, hyperparameter optimization)")
print("=" * 80)

results = text_search(datatalk_index, "how to improve models", top_k=5)
print(f"\nFound {len(results)} results:\n")

for i, result in enumerate(results, 1):
    print(f"{i}. [Score: {result.get('score', 0):.3f}] {result['title']}")
    print(f"   Preview: {result['content'][:150]}...")
    print()

print("❌ Text search limitation: No exact keyword overlap with 'fine-tuning' or 'optimization'")
print("   The query uses different words than the actual content.")

In [ ]:
# Limitation 2: Conceptual query - "understanding data quality"
print("=" * 80)
print("Limitation Query: 'understanding data quality'")
print("(Looking for content about data validation, metrics, monitoring)")
print("=" * 80)

results = text_search(evidently_index, "understanding data quality", top_k=5)
print(f"\nFound {len(results)} results:\n")

for i, result in enumerate(results, 1):
    print(f"{i}. [Score: {result.get('score', 0):.3f}] {result['title']}")
    print(f"   Preview: {result['content'][:150]}...")
    print()

print("❌ Text search limitation: Query is conceptual, not using specific metric names")
print("   Documents might discuss quality without using the word 'understanding'.")

### Why Text Search Fails Here

Text search requires **exact keyword overlap**:
- "how to improve models" doesn't contain "fine-tuning" or "optimization"
- "understanding data quality" is too general - documents use specific terms like "drift detection" or "metric calculation"

**This is why we need semantic search** (Day 4): Vector embeddings can match "improve models" with "fine-tuning" because they understand the semantic relationship, even when keywords don't overlap.

## Day 3 Learnings Summary

### What is TF-IDF?

TF-IDF (Term Frequency-Inverse Document Frequency) scores words based on:
- **Term Frequency (TF)**: How often a word appears in a document
- **Inverse Document Frequency (IDF)**: How rare a word is across all documents

Words that appear frequently in a document but rarely across all documents get high scores. Common words like "the" or "is" get low scores.

### Why Field Boosting Matters

With `boost_dict={"title": 2.0, "content": 1.0}` at search time:
- A title match counts 2x more than a content match
- If a query term appears in the title, that document ranks higher
- This is especially useful for documentation where titles are highly descriptive

### When Text Search Works Best

✅ **Exact terminology**: "TF-IDF", "machine learning", "data engineering"  
✅ **Acronyms**: "API", "FAQ", "ML"  
✅ **Proper names**: "Docker", "Evidently", course names  
✅ **Technical terms**: "monitoring", "data drift", "embeddings"  

### When Text Search Fails

❌ **Paraphrases**: "how to improve models" ≠ "fine-tuning techniques"  
❌ **Conceptual queries**: "understanding data quality" ≠ specific metric names  
❌ **Synonyms**: "buy" ≠ "purchase", "start" ≠ "begin"  
❌ **Different vocabulary**: User query uses different words than document content  

### Preview: Day 4 - Semantic Search

Semantic search (vector search) solves the paraphrase problem:
- Converts text to embeddings (numerical vectors)
- Measures semantic similarity, not just keyword overlap
- "improve models" matches "fine-tuning" because embeddings capture meaning
- Best for conceptual queries, question answering, and when users don't know exact terminology

Day 5 will combine both approaches (hybrid search) for the best of both worlds.